In [139]:
import re
import json
import torch
from typing import List, Dict, Set
from tqdm.auto import tqdm
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, NewsMorphTagger, Doc, MorphVocab
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

In [140]:
CONFIDENCE_THRESHOLD = 0.3
FACT_MATCHING_THRESHOLD = 0.81

In [141]:
LABEL_TARGET = "specific brand product or business unit"
LABEL_NOISE = "general financial term or abstract concept"

In [142]:
segmenter = Segmenter()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)
morph_tagger = NewsMorphTagger(emb)
morph_vocab = MorphVocab()

INFO:pymorphy2.opencorpora_dict.wrapper:Loading dictionaries from c:\Users\Kirill\anaconda3\envs\common_classic\Lib\site-packages\pymorphy2_dicts_ru\data
INFO:pymorphy2.opencorpora_dict.wrapper:format: 2.4, revision: 417127, updated: 2020-10-11T15:05:51.070345


In [143]:
classifier = pipeline("zero-shot-classification", 
                      model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", 
                      device=-1)

Device set to use cpu


In [144]:
embedder = SentenceTransformer('intfloat/multilingual-e5-base')

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: intfloat/multilingual-e5-base


In [ ]:
def clean_text(text: str) -> str:
    """Убирает пунктуацию из названия сущности."""
    return re.sub(r'[«»"():]', '', text).strip()

def extract_candidates(text: str) -> Set[str]:
    """
    Первичный сбор кандидатов через NER и эвристику (Заглавная буква + Существительное).
    """
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)
    doc.tag_morph(morph_tagger)
    
    candidates = set()
    
    for span in doc.spans:
        if span.type == 'ORG':
            span.normalize(morph_vocab)
            candidates.add(span.normal)

    for token in doc.tokens:
        token.lemmatize(morph_vocab)
        if token.text[0].isupper() and len(token.text) > 2:
            if token.pos in ['NOUN', 'PROPN']:
                candidates.add(token.lemma.capitalize())
    
    cleaned_candidates = {clean_text(c) for c in candidates}
    return {c for c in cleaned_candidates if len(c) > 2}

def filter_entities(candidates: Set[str]) -> List[str]:
    """
    Фильтрация кандидатов через Zero-Shot классификатор (вместо якорей).
    """
    valid_entities = []
    labels = [LABEL_TARGET, LABEL_NOISE]
    
    candidates_list = list(candidates)
    if not candidates_list:
        return []

    for cand in tqdm(candidates_list, desc="Validating entities"):
        res = classifier(cand, candidate_labels=labels, multi_label=False)
        
        predicted_label = res['labels'][0]
        score = res['scores'][0]
        
        if predicted_label == LABEL_TARGET and score > CONFIDENCE_THRESHOLD:
            valid_entities.append(cand)
            
    return valid_entities

def match_facts_to_entities(text: str, entities: List[str]) -> Dict[str, List[str]]:
    """
    Ищет в тексте предложения с цифрами, семантически связанные с найденными сущностями.
    """
    if not entities:
        return {}

    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n', text) if len(s.strip()) > 20]
    fact_sentences = [s for s in sentences if re.search(r'\d', s)]
    
    if not fact_sentences:
        return {e: [] for e in entities}

    entity_queries = [f"query: финансовые результаты {e}" for e in entities]
    passage_docs = [f"passage: {s}" for s in fact_sentences]
    
    proj_embs = embedder.encode(entity_queries, convert_to_tensor=True, show_progress_bar=False)
    sent_embs = embedder.encode(passage_docs, convert_to_tensor=True, show_progress_bar=True)
    
    scores = util.cos_sim(proj_embs, sent_embs)
    
    raw_report = {}
    
    for i, entity in enumerate(entities):
        hits = torch.where(scores[i] > FACT_MATCHING_THRESHOLD)[0]
        facts = []
        
        for idx in hits:
     
            root = entity[:-1].lower() if len(entity) > 4 else entity.lower()
            if root in fact_sentences[idx].lower():
                facts.append(fact_sentences[idx])
        
        if facts:
            raw_report[entity] = list(set(facts))
            
    return raw_report

def deduplicate_keys(report: Dict[str, List[str]]) -> Dict[str, List[str]]:
    """
    Объединяет ключи, отличающиеся только регистром (Yandex Cloud = yandex cloud).
    """
    clean_report = {}
    seen_map = {} 
    
    for entity, facts in report.items():
        low_key = entity.lower()
        if low_key in seen_map:
            primary_key = seen_map[low_key]
            clean_report[primary_key].extend(facts)
            clean_report[primary_key] = list(set(clean_report[primary_key]))
        else:
            seen_map[low_key] = entity
            clean_report[entity] = facts
            
    return clean_report

def process_report(text: str) -> Dict[str, List[str]]:
    """Основной пайплайн обработки."""
    # 1. NER + Heuristics
    candidates = extract_candidates(text)
    
    # 2. Classifier Filtering
    valid_entities = filter_entities(candidates)
    
    # 3. Fact Matching
    report = match_facts_to_entities(text, valid_entities)
    
    # 4. Cleanup
    return deduplicate_keys(report)

In [146]:
FILE_PATH = r"./report.txt"
OUTPUT_FILE = "result.json"

In [ ]:


try:
    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        content = f.read()
    
    print(f"Processing {FILE_PATH}...")
    final_data = process_report(content)
    
    print(json.dumps(final_data, ensure_ascii=False, indent=4))
    
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(final_data, f, ensure_ascii=False, indent=4)
    print(f"Saved to {OUTPUT_FILE}")

except FileNotFoundError:
    print(f"Error: File {FILE_PATH} not found.")
except Exception as e:
    print(f"Error: {e}")

Processing ./report.txt...


Batches: 100%|██████████| 6/6 [00:08<00:00,  1.34s/it]

{
    "Пэй": [
        "16 Общий оборот (GMV) финансовых сервисов — совокупный объем всех покупок пользователей с использованием продуктов Яндекс Пэй."
    ],
    "МКПАО": [
        "В рамках утвержденной советом директоров МКПАО «ЯНДЕКС» программы биржевых облигаций 9 сентября был размещен второй выпуск облигаций на сумму 25 млрд рублей с ежемесячной выплатой купона с фиксированной ставкой в 13,5%.",
        "МКПАО «ЯНДЕКС» (MOEX: YDEX), ведущая частная IT-компания, которая создает и развивает сервисы и технологии мирового уровня для пользователей и для бизнеса, объявляет неаудированные финансовые результаты за третий квартал 2025 года."
    ],
    "Яндекс Лавка": [
        "14 Валовой оборот (GTV) сервисов Электронной коммерции — совокупная стоимость всех проданных, доставленных и оплаченных товаров на платформах Яндекс Маркет и Яндекс Лавка, а также совокупная стоимость заказов, доставленных сервисами Яндекс Еда и Деливери (доставка продуктов из магазинов и готовой еды из ресторанов